# Beyond RAG — VINE pipeline (Colab)

**v5 — text rebuild + artwork figure crops.**

What changed since the last run:

* **Figure crops come from the artwork layer.** Measured on the old crops, 559
  of 560 figures had their caption *outside* the crop, and crops ran on into
  body prose (Figure 8D-3's real extent is 137 pt tall; its crop was 478 pt and
  covered five paragraphs). Now 558/558 include the caption, 4 hold any body
  prose, median crop area 77% of the old one. Tables are unchanged — they were
  already bounded by their font layer.
* **Parser fixes.** 191 sections were being cut short by body lines beginning
  "Section 2A.12 contains …"; 1C.02 Definitions ended at definition 125 and now
  reaches 295. All 951 sections have chunks (was 930). Page headers, footers
  and "Rev. 1" marks are gone. Every chunk used to say "Part 9".
* **New chunk kinds.** Lists split one chunk per item (1,920 items, each
  keeping its lead-in); the Part 6 "Notes for Figure 6P-N" pages (469 items,
  those pages went from 6% to 100% covered); the Appendices; and notes printed
  inside figures and tables (583 chunks, 99 carrying shall/should).
* **Graph.** 252 real sections used to stay as unresolved `SectionRef` stubs —
  now 953 proper Section nodes. New `Paragraph` nodes, `part_of_paragraph` and
  `note_on` edges.

Totals: **9,169 chunks** (was 5,812), **12,090 nodes / 26,207 edges**.

Still broken, on purpose, to fix next: the sparse (keyword) leg is empty —
FlagEmbedding ≥1.4 passes `dtype=` that transformers 4.54.1 rejects, and the
code falls back to dense-only.

Run order: **0 → 1 → 1.2 → 2 → 2.2 → 3 → 4**.

## 0. Setup

In [ ]:
# COLAB ONLY — skip this cell if running on HPRC.
import sys, os

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# ── WHERE EVERYTHING LIVES ─────────────────────────────────────────────────
# Data (PDF, cache, figures, page images, HF cache, snapshots, runs) lives on
# Drive. The repo is cloned to local disk and is disposable.
#
# MRAG_BASE_DIR overrides the default in config.py. It must be set BEFORE any
# mrag import, and it must be an env var so subprocesses (ingest_v4.py, run via
# !python) inherit it — sys.modules detection does not cross process boundaries.
DRIVE_DIR = "/content/drive/MyDrive/Beyond_RAG"       # <- data on Drive
REPO_DIR  = "/content/Beyond_RAG_repo"                # <- code, local, disposable
REPO_URL  = "https://github.com/hannanazad/Beyond_RAG.git"

os.environ["MRAG_ENV"]      = "colab"
os.environ["MRAG_BASE_DIR"] = DRIVE_DIR
os.makedirs(DRIVE_DIR, exist_ok=True)

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

sys.path.insert(0, REPO_DIR)
print(f"\ndata : {DRIVE_DIR}")
print(f"code : {REPO_DIR}")

# Step 0 — poppler-utils.
# Colab ships a PARTIAL poppler install: pdftotext and pdftoppm are present
# but pdftohtml is NOT. mrag/font_index.py needs pdftohtml to read the font
# a caption is set in; without it, caption anchors cannot be validated and
# body-text mentions get cropped as figures.
!apt-get install -y -qq poppler-utils
!which pdftotext pdftoppm pdftohtml pdffonts

# Step 1 — torch matched to Colab's CUDA 12.4.
# Colab ships torch 2.5.1, but transformers >=4.51 enforces CVE-2025-32434 and
# refuses to load .bin checkpoints on torch <2.6. BGE-M3 ships .bin.
!pip install -q --index-url https://download.pytorch.org/whl/cu124 \
    torch==2.6.0 torchvision==0.21.0

# Step 2 — everything else
!pip install -q -r $REPO_DIR/requirements.txt

# Step 3 — pip's bulk resolver does not enforce CEILINGS when an installed
# version already satisfies the lower bound. Colab ships newer packages than
# this pipeline can use, so pin them explicitly.
!pip install -q --no-deps --force-reinstall \
    "transformers>=4.49,<4.55" \
    "huggingface_hub>=0.34,<0.35" \
    "tokenizers>=0.21,<0.22" \
    "torchao>=0.13,<0.14"

!python -c "from transformers import PreTrainedModel; print('transformers import OK')"

In [ ]:
# API keys from Colab Secrets (key icon, left sidebar).
# Keys authenticate only; select the model with CFG.set_vlm_model(...).
from google.colab import userdata
import os

def _load(env_name, *secret_names):
    for s in secret_names:
        try:
            os.environ[env_name] = userdata.get(s)
            print(f"{env_name}: loaded (secret {s!r})")
            return
        except Exception:
            continue
    print(f"{env_name}: no matching secret (skipped)")

_load("DASHSCOPE_API_KEY", "DASHSCOPE_API_KEY", "QWEN")
_load("ANTHROPIC_API_KEY", "ANTHROPIC_API_KEY")
#_load("GEMINI_API_KEY",    "GEMINI_API_KEY")

### 0.1 Moving from `Drive/MyDrive/MRAG`

In [ ]:
import os
from pathlib import Path

old = Path("/content/drive/MyDrive/MRAG")
new = Path(os.environ["MRAG_BASE_DIR"])

def survey(p):
    if not p.exists():
        return "does not exist"
    bits = []
    for name in ("mmrag_cache_v3", "figures", "page_images", "hf_cache"):
        d = p / name
        bits.append(f"{name}={'yes' if d.exists() else 'no'}")
    pdfs = list(p.glob("*.pdf"))
    bits.append(f"pdf={pdfs[0].name if pdfs else 'MISSING'}")
    bits.append(f"snapshot={'yes' if (p/'qdrant_db.tar').exists() else 'no'}")
    return "  ".join(bits)

print(f"OLD  {old}\n     {survey(old)}\n")
print(f"NEW  {new}\n     {survey(new)}\n")

if old.exists() and not (new / "mmrag_cache_v3").exists():
    print("ACTION: rename MRAG -> Beyond_RAG in the Drive web UI, then re-run cell 0.")
    print("        Copying through Colab works but is slow — page_images alone is ~3 GB.")
elif (new / "mmrag_cache_v3").exists():
    print("Beyond_RAG already holds the cache. Nothing to do.")
else:
    print("Neither folder has a cache. Upload the MUTCD PDF to Beyond_RAG and ingest.")

## 1. Sanity check

In [ ]:
import os, sys
REPO_DIR = "/content/Beyond_RAG_repo"
sys.path.insert(0, REPO_DIR if os.path.isdir(REPO_DIR) else ".")

from mrag.config import CFG

print("Environment :", CFG.environment)
print("Base dir    :", CFG.base_dir, "| exists:", CFG.base_dir.exists())
print("PDF path    :", CFG.pdf_path, "| exists:", CFG.pdf_path.exists())
print("Cache dir   :", CFG.cache_dir)
print("Qdrant dir  :", CFG.qdrant_dir, "  (local SSD, snapshotted to Drive)")
print("HF cache    :", CFG.hf_home)
print("VLM provider:", CFG.vlm_provider)
print("VLM model   :", CFG.vlm_model_api if CFG.vlm_provider == "api" else CFG.vlm_model)
print("API key set :", bool(os.environ.get(CFG.api_key_env_var)))
print()
print("Image budget: max_sheets_per_figure =", CFG.max_sheets_per_figure,
      "| max_images_total =", CFG.max_images_total,
      "| max_page_images =", CFG.max_page_images)

assert str(CFG.base_dir).endswith("Beyond_RAG"), \
    f"base_dir is {CFG.base_dir} — MRAG_BASE_DIR did not take. Re-run cell 0."

try:
    import torch
    print("GPU         :", torch.cuda.get_device_name(0))
except Exception as e:
    print("GPU         : none —", e)

assert CFG.pdf_path.exists(), (
    f"No PDF at {CFG.pdf_path} or anywhere in {CFG.base_dir}. "
    f"Upload the MUTCD PDF there first."
)

### 1.1 Verify the caption font filter

The extractor used to treat any `Figure X-Y.` as a caption, including
mid-sentence in body text — 79 crops in the previous corpus were paragraphs
cropped that way, and 60 of them overwrote their node's caption and page.

MUTCD sets body text in **Times** and captions in **Helvetica**, so the font
settles it. Measured over all 663 pages holding a crop: **79 of 79 junk
rejected, 648 of 648 good kept**.

All six checks below must print OK. If `pdftohtml` is missing this raises with
an install command rather than failing silently.

In [ ]:
from mrag.font_index import get_font_index

fi = get_font_index(str(CFG.pdf_path))        # one pdftohtml pass, ~15s, cached

checks = [(94, 'Figure', '2A-4', True),   (91,  'Figure', '2A-4', False),
          (109,'Table',  '2B-1', True),   (108, 'Table',  '2B-1', False),
          (250,'Figure', '2D-2', False),  (662, 'Table',  '3G-1', True)]
for pno, kind, cid, expect in checks:
    got = fi.has_caption_font(pno, kind, cid)
    print(f"{'OK ' if got == expect else 'FAIL'} p{pno:<5} {kind} {cid:<6} -> {got}")

### 1.2 Force the v5 rebuild (run ONCE)

The ingest skips any step whose cache already exists, and the restore cell in
section 2 will happily bring back a Qdrant snapshot built from the old chunks.
So the stale artefacts have to go first.

`figures.jsonl` is detected automatically — the new records are tagged
`caption_below_v5`, so a v2 file is archived and re-extracted without being
deleted. The rest is removed here. Everything removed is rebuilt by the ingest;
`.bak` copies are kept where the ingest makes them.

In [ ]:
from pathlib import Path
import shutil, json

cache = CFG.cache_dir
# These three invalidate THEMSELVES, so they are NOT listed below:
#   figures.jsonl   tagged caption_below_v5; a v2 file is archived + re-extracted,
#                   and that also clears figures_dense.npy and colqwen_figures/
#   graph.gpickle   schema_version 3; a v2 graph is archived and rebuilt
#   the figures/ folder of PNGs  crop file names are identical, so the new crops
#                   overwrite the old ones in place
stale = [
    cache / "chunks.jsonl",              # reparsed: notes, list items, appendices
    cache / "chunks_v4.stamp",           # old stamp name
    cache / "chunks_v5.stamp",
    cache / "chunks_dense.npy",          # chunk text changed
    cache / "chunks_sparse.json",
    cache / "text_embeddings_manifest.json",
    CFG.base_dir / "qdrant_db.tar",      # snapshot built from the old chunks
]
# Belt and braces: the figure step clears these itself when it re-extracts.
colqwen_figs = cache / "colqwen_figures"
for extra in (cache / "figures_dense.npy",):
    if extra.exists():
        extra.unlink(); print("removed:", extra.name)

print("removing:")
for p in stale:
    if p.exists():
        print("  ", p.name, f"({p.stat().st_size/1e6:.1f} MB)")
        p.unlink()
    else:
        print("  ", p.name, "(absent)")
if colqwen_figs.exists():
    n = len(list(colqwen_figs.glob("*.npy")))
    shutil.rmtree(colqwen_figs)
    print(f"   colqwen_figures/ ({n} crop vectors)")

# page vectors are NOT deleted: the page PNGs are unchanged, and re-encoding
# 1,162 pages costs ~4 GPU-minutes for no benefit
pages = cache / "colqwen_pages"
print("kept:", pages.name, f"({len(list(pages.glob('*.npy'))) if pages.exists() else 0} page vectors)")

f = CFG.figures_jsonl
if f.exists():
    first = json.loads(open(f).readline())
    v = str(first.get("extraction_method", "?"))
    print(f"\nfigures.jsonl is {v!r} -> "
          + ("current, will be reused" if v.startswith("caption_below_v5")
             else "stale, the ingest will archive it and re-extract"))

import pickle
gp = CFG.graph_pickle
if gp.exists():
    try:
        sv = pickle.load(open(gp, "rb")).graph.get("schema_version")
    except Exception:
        sv = "unreadable"
    print(f"graph.gpickle is schema {sv} -> "
          + ("current, will be reused" if sv == 3
             else "stale, the ingest will archive it and rebuild"))

## 2. Build or restore the vector store

Restores a Drive snapshot if one exists. Otherwise runs a full ingest.

⚠ **A fresh ingest rewrites every figure and table crop.** Back up
`figures/` first if it holds hand-corrected images.

In [ ]:
import shutil, tarfile, json
from pathlib import Path
from qdrant_client import QdrantClient

REPO = "/content/Beyond_RAG_repo"
local_qdrant     = CFG.qdrant_dir
drive_qdrant_tar = CFG.base_dir / "qdrant_db.tar"

def _qdrant_ok(path: Path) -> bool:
    if not path.exists():
        return False
    try:
        c = QdrantClient(path=str(path))
        try:
            return CFG.coll_chunks in {col.name for col in c.get_collections().collections}
        finally:
            c.close()
    except Exception:
        return False

def _v5_done() -> bool:
    """figures.jsonl must come from the artwork-bounds extractor."""
    try:
        with open(CFG.figures_jsonl) as f:
            first = json.loads(next(f))
        return str(first.get("extraction_method", "")).startswith("caption_below_v5")
    except Exception:
        return False

def _newest_mtime(path: Path) -> float:
    return max((p.stat().st_mtime for p in path.rglob("*") if p.is_file()), default=0.0)

def _store_fresh() -> bool:
    """The LOCAL store must have been built from the chunks that are on disk
    now. Checking only that a collection exists let a store survive a cache
    wipe: the cell reported "nothing to do" while chunks.jsonl was missing."""
    if not (_qdrant_ok(local_qdrant) and CFG.chunks_jsonl.exists()):
        return False
    return _newest_mtime(local_qdrant) >= CFG.chunks_jsonl.stat().st_mtime

def _snapshot_fresh() -> bool:
    """A Drive snapshot older than chunks.jsonl was built from different text.
    Restoring it silently shipped the previous corpus, so refuse it."""
    if not (drive_qdrant_tar.exists() and CFG.chunks_jsonl.exists()):
        return False
    return drive_qdrant_tar.stat().st_mtime >= CFG.chunks_jsonl.stat().st_mtime

if _store_fresh() and _v5_done():
    print(f"Qdrant populated at {local_qdrant} and figures are v5. Nothing to do.")

elif _snapshot_fresh() and _v5_done():
    print(f"Restoring snapshot from {drive_qdrant_tar} "
          f"({drive_qdrant_tar.stat().st_size/1e6:.0f} MB)...")
    shutil.rmtree(local_qdrant, ignore_errors=True)
    local_qdrant.mkdir(parents=True, exist_ok=True)
    with tarfile.open(drive_qdrant_tar, "r") as tar:
        tar.extractall(local_qdrant.parent)
    assert _qdrant_ok(local_qdrant), "Extracted but collection not found."
    print("Local Qdrant verified OK.")

else:
    print("No usable store. Reasons:")
    print(f"  local Qdrant populated : {_qdrant_ok(local_qdrant)}   ({local_qdrant})")
    print(f"  local store matches chunks.jsonl : {_store_fresh()}")
    print(f"  Drive snapshot usable  : {_snapshot_fresh()}   ({drive_qdrant_tar})")
    print(f"  figures.jsonl is v5    : {_v5_done()}   ({CFG.figures_jsonl})")
    print(f"  chunks.jsonl exists    : {CFG.chunks_jsonl.exists()}")
    print(f"  graph.gpickle exists   : {CFG.graph_pickle.exists()}")
    print()
    if CFG.chunks_jsonl.exists() and not drive_qdrant_tar.exists():
        print("  Cache is present but there is no usable Qdrant snapshot.")
        print("  If figures.jsonl is still v2 the crops are re-extracted too:")
        print("  budget ~45 min for the full v5 rebuild, ~10 min if the crops")
        print("  are already v5 and only text + embeddings re-run.")
    print()
    print("Running ingest.")
    !cd $REPO && python scripts/ingest_v4.py
    assert _qdrant_ok(local_qdrant), "Ingest finished but Qdrant is empty — check the log."
    assert _v5_done(),               "Ingest ran but figures.jsonl is not v5 — check the log."
    print("Ingest verified OK.")

### 2.1 Snapshot Qdrant back to Drive

In [ ]:
if CFG.environment == "colab":
    local_qdrant     = CFG.qdrant_dir
    drive_qdrant_tar = CFG.base_dir / "qdrant_db.tar"

    if local_qdrant.exists():
        print(f"Snapshotting {local_qdrant} -> {drive_qdrant_tar} ...")
        drive_qdrant_tar.parent.mkdir(parents=True, exist_ok=True)
        with tarfile.open(drive_qdrant_tar, "w") as tar:
            tar.add(local_qdrant, arcname=local_qdrant.name)
        print(f"Done ({drive_qdrant_tar.stat().st_size/1e6:.0f} MB).")
    else:
        print("No local Qdrant to snapshot.")

## 2.2 Verify the v5 build

Run this straight after the ingest, **before** taking a snapshot. Four checks:
figure crops, chunk mix, corpus coverage, and graph shape. Each prints the
number the rebuild produced here so a mismatch is obvious.

### 2.2.1 Figure crops now contain their caption

In [ ]:
import json, pymupdf
from mrag.figure_bounds import page_layers

figs = [json.loads(l) for l in open(CFG.figures_jsonl) if l.strip()]
doc  = pymupdf.open(str(CFG.pdf_path))
kinds = {}
for f in figs: kinds[f["kind"]] = kinds.get(f["kind"], 0) + 1
print("crops:", len(figs), kinds)

cap_in = body_in = 0
figs_only = [f for f in figs if f["kind"] == "Figure"]
for f in figs_only:
    page = doc.load_page(f["page_pdf"] - 1)
    box  = pymupdf.Rect(f["bbox"])
    _, body, caps = page_layers(page)
    key = "Figure " + f["canonical_id"]
    cr = next((r for k, r in caps if k == key), None)
    if cr and box.contains(cr): cap_in += 1
    if any(box.contains(pymupdf.Point((r.x0+r.x1)/2, (r.y0+r.y1)/2)) for r in body): body_in += 1
print(f"caption inside the crop : {cap_in}/{len(figs_only)}   (expected ~558/560; old build: 1/560)")
print(f"crops holding body prose: {body_in}                    (expected ~4)")
doc.close()

Look at a figure that was badly cropped before — caption cut off and five paragraphs of Section 8D.15 swept in.

In [ ]:
from IPython.display import display, Image as IPImage
import json
figs = [json.loads(l) for l in open(CFG.figures_jsonl) if l.strip()]
for fid in ("8D-3", "6G-1", "2G-15"):
    for f in [x for x in figs if x["canonical_id"] == fid and x["kind"] == "Figure"]:
        print(f"Figure {fid}  pdf p{f['page_pdf']}  bbox={[round(v) for v in f['bbox']]}")
        display(IPImage(filename=f["image_path"], width=620))

### 2.2.2 What the corpus is made of

In [ ]:
import json, collections
rows = [json.loads(l) for l in open(CFG.chunks_jsonl) if l.strip()]
print("chunks:", len(rows), "   (old build: 5,812 — expected 9,169)")
print("by source     :", dict(collections.Counter(r.get("source", "paragraph") for r in rows)))
print("by rule type  :", dict(collections.Counter(r["content_type"] for r in rows)))
print("sections with chunks:", len({r["section_id"] for r in rows}), "(expected 953, incl. A1/A2)")
inf = [r for r in rows if r.get("authority_inferred")]
print("authority inferred from the verb:", len(inf),
      "|", dict(collections.Counter(r["content_type"] for r in inf)))
print("  of those, carrying shall/should:",
      sum(1 for r in inf if r["content_type"] in ("Standard", "Guidance")))

print("\nleftovers that should all be zero:")
for pat in ("December 2025", "MUTCD 11th Edition", "Rev. 1", "intentionally left blank"):
    print(f"   chunks containing {pat!r}: {sum(1 for r in rows if pat in r['text'])}")
print("   chunks whose Part is wrong:",
      sum(1 for r in rows if r["section_id"][0].isdigit()
          and not (r.get("part") or "").startswith(f"Part {r['section_id'][0]} ")))

Three provisions that the old parser lost entirely.

In [ ]:
import json
rows = [json.loads(l) for l in open(CFG.chunks_jsonl) if l.strip()]
by_id = {r["chunk_id"]: r for r in rows}

defs = [r for r in rows if r["section_id"] == "1C.02" and r.get("source") == "list_item"]
print(f"1C.02 definitions: {len(defs)}   (old build stopped at 125)")
print("   last one:", defs[-1]["text"][-110:], "\n")

for cid in ("MUTCD11e_2B48_Standard_01",        # section had no chunks at all
            "MUTCD11e_6P01_TA14_Standard_04",   # Part 6 notes page
            "MUTCD11e_TBLNOTE_6B-4_01",         # table note: the L/W/S key
            "MUTCD11e_TBLNOTE_3G-1_03",         # table note that caps the table
            "MUTCD11e_A1_Standard_07"):         # appendix
    r = by_id.get(cid)
    print(f"{cid}\n   {'MISSING' if not r else r['content_type'] + ' | ' + r['text'][:150]}\n")

### 2.2.3 Coverage against the manual

Judged against the MUTCD itself, not against the benchmark: what share of the
manual's body text (Times face) ends up in a chunk. The front matter is title
pages, credits and contents lists; the 0.77% left in the main body is the
per-chapter "Section Organization" index lists. Both are navigation, excluded
on purpose.

In [ ]:
!cd /content/Beyond_RAG_repo && python scripts/coverage_audit.py . "{CFG.pdf_path}" "{CFG.figures_jsonl}"

### 2.2.4 Graph shape

In [ ]:
import collections
from mrag.kg import read as kg_read
g = kg_read(CFG.graph_pickle)
print("nodes:", g.number_of_nodes(), " edges:", g.number_of_edges(),
      "  (old build: 8,323 / 18,157 — expected 12,090 / 26,207)")
print("node kinds :", dict(collections.Counter(d.get("kind") for _, d in g.nodes(data=True))))
print("edge labels:", dict(collections.Counter(d.get("label") for *_, d in g.edges(data=True))))

secs = [(n, d) for n, d in g.nodes(data=True) if n.startswith("section:")]
stub = [n for n, d in secs if d.get("kind") != "Section"]
print(f"\nsection nodes: {len(secs)} | still placeholders: {len(stub)}  "
      f"(old build: 252) {sorted(stub)[:8]}")
print("sections with a title:", sum(1 for _, d in secs if d.get("title")))
print("notes linked to their figure (note_on):",
      sum(1 for *_, d in g.edges(data=True) if d.get("label") == "note_on"))
print("list items linked to their paragraph:",
      sum(1 for *_, d in g.edges(data=True) if d.get("label") == "part_of_paragraph"))

### 2.3 Repair colliding chunk IDs

Report only. The v5 parser gives every chunk a unique id (checked over all
9,169), so this should find nothing. Run it if you want the confirmation.


In [ ]:
REPO = "/content/Beyond_RAG_repo"

# Report only — nothing is written.
!cd $REPO && python scripts/repair_chunk_ids.py --cache "{CFG.cache_dir}"

# To actually rewrite chunks.jsonl (a .prerepair.bak is kept), uncomment:
# !cd $REPO && python scripts/repair_chunk_ids.py --cache "{CFG.cache_dir}" --apply

## 3. Initialise the pipeline

In [ ]:
import logging, os
logging.basicConfig(level=logging.INFO, format='%(name)s - %(message)s')

if CFG.vlm_provider == "api" and not os.environ.get(CFG.api_key_env_var):
    print(f"NOTE: no key in {CFG.api_key_env_var!r} for {CFG.vlm_model_api!r}. "
          f"Init will proceed; load the key or switch with CFG.set_vlm_model(...).")

from mrag.ask import init_pipeline
pipeline = init_pipeline()
print("VLM loaded :", pipeline.vlm.loaded_name if pipeline.vlm else "none")
print("KG         :", pipeline.kg.g.number_of_nodes(), "nodes,",
                      pipeline.kg.g.number_of_edges(), "edges")

### 3.1 Choose the generation model

Every row you intend to compare must use the **same** model.

In [ ]:
# ── List every VLM the config knows about, grouped by provider ──────────────
import os
from mrag.config import (
    CFG, VLM_API_MODELS, VLM_PROVIDERS, provider_of_model,
    VLM_TEXT_ONLY_ALIASES, VLM_VISION_UNVERIFIED,
)

# key present?  ->  can you actually call it
have_key = {p: bool(os.environ.get(v["env_var"])) for p, v in VLM_PROVIDERS.items()}
print("API keys loaded:", {p: ("yes" if k else "NO") for p, k in have_key.items()})
print("Currently selected:", CFG.vlm_model_api, f"[{provider_of_model(CFG.vlm_model_api)}]")
print()

# one row per distinct model id, collecting the aliases that point at it
by_model = {}
for alias, mid in VLM_API_MODELS.items():
    by_model.setdefault(mid, []).append(alias)

for prov in ("anthropic", "gemini", "dashscope"):
    rows = [(m, a) for m, a in by_model.items() if provider_of_model(m) == prov]
    if not rows:
        continue
    print(f"── {prov.upper()}   (key: {'yes' if have_key[prov] else 'NO'})")
    for mid, aliases in sorted(rows):
        flags = []
        if mid in VLM_VISION_UNVERIFIED:
            flags.append("VISION UNVERIFIED")
        if any(a in VLM_TEXT_ONLY_ALIASES for a in aliases):
            flags.append("TEXT ONLY - will 400 on images")
        if mid == CFG.vlm_model_api:
            flags.append("<< SELECTED")
        note = ("   " + " | ".join(flags)) if flags else ""
        print(f"   {'  '.join(sorted(aliases)):<42} -> {mid}{note}")
    print()

print("Select with:  CFG.set_vlm_model('fast_claude')")
print("A raw model id works too:  CFG.set_vlm_model('claude-sonnet-5')")

In [ ]:
# ── Pick a model ────────────────────────────────────────────────────────────
resolved = CFG.set_vlm_model("frontier_claude")      # cheap, for smoke tests

print("model    :", resolved)
#print("provider :", provider_of_model(resolved))
print("key set  :", bool(os.environ.get(CFG.api_key_env_var)), f"({CFG.api_key_env_var})")

# The pipeline reads CFG at call time, so no re-init needed.
#_ = ask("What shape and colour is a STOP sign?")

## 4. Ask

In [ ]:
from mrag.ask import ask

#_ = ask("What Speed Limit (R2-1) sign sizes are shown in Table 2B-1 for conventional single-lane, conventional multi-lane, expressway, and freeway applications?")

### 4.1 Multi-sheet figures

A canonical figure or table can span several sheets. Retrieval emits them all;
the prompt builder used to show only sheet 1. 116 of 553 canonical entities are
multi-sheet, hiding 174 sheets.

Allocation is **round-robin** — sheet 1 of every figure, then sheet 2 of every
figure — because taking them figure-by-figure let one 8-sheet table starve
everything after it.

In [ ]:
import json, collections
figs = [json.loads(l) for l in open(CFG.figures_jsonl) if l.strip()]
canon = collections.defaultdict(list)
for f in figs:
    canon[(f["kind"], f["canonical_id"])].append(f)      # key on BOTH — ids collide across kinds

multi = {k: v for k, v in canon.items() if len(v) > 1}
print(f"canonical entities : {len(canon)}")
print(f"multi-sheet        : {len(multi)}")
print(f"sheets beyond first: {sum(len(v) - 1 for v in multi.values())}")
print(f"widest             : {sorted(((len(v), k[1]) for k, v in multi.items()), reverse=True)[:5]}")
print()
print(f"caps: sheets/figure={CFG.max_sheets_per_figure}  "
      f"images/request={CFG.max_images_total}  pages reserved={CFG.max_page_images}")
covered = sum(1 for v in canon.values() if len(v) <= CFG.max_sheets_per_figure)
print(f"entities fully shown under the per-figure cap: {covered}/{len(canon)}")

# ── allocation check ───────────────────────────────────────────────────────
# Selection is ROUND-ROBIN: sheet 1 of every figure, then sheet 2 of every
# figure, and so on. Greedy allocation let one 8-sheet table eat the budget and
# starve later figures, which cost more than it gained (TB009 went from full
# credit to abstaining). Every retrieved figure must keep at least sheet 1.
def simulate(sheet_counts, cap=None, per_fig=None, pages=1, reserve=None):
    cap     = cap or CFG.max_images_total
    per_fig = per_fig or CFG.max_sheets_per_figure
    reserve = CFG.max_page_images if reserve is None else reserve
    per = [list(range(min(n, per_fig))) for n in sheet_counts]
    budget = max(0, cap - min(pages, reserve))
    picked, depth = [], 0
    while len(picked) < budget and any(len(s) > depth for s in per):
        for fi, s in enumerate(per):
            if depth < len(s) and len(picked) < budget:
                picked.append((fi, depth))
        depth += 1
    shown = len({fi for fi, _ in picked})
    return len(picked), shown, len(sheet_counts)

for label, counts in [("TB009-like: 8-sheet table + 2 figures", [8, 6, 3]),
                      ("one 8-sheet table alone",               [8]),
                      ("five single-sheet figures",             [1, 1, 1, 1, 1])]:
    n, shown, total = simulate(counts)
    ok = "OK" if shown == total else "STARVED"
    print(f"  {label:<40} {n:>2} imgs, {shown}/{total} figures  {ok}")

In [ ]:
# TB009 is the regression to watch. It asks for four R2-1 sign sizes; gold
# evidence is Table 2B-1, which has 8 sheets. Under greedy allocation the model
# ABSTAINED — 0.00 correctness where it previously had full credit.
#
# Expect: several images, each labelled "[sheet N of M]", and an answer that
# actually lists the sizes.
_ = ask("What are the four sizes listed for the R2-1 Speed Limit sign?",
        show_scores=True)

### 4.2 Figure relevance filter

Figures arriving by graph link have no score and were never checked against the
query, so off-topic tables ate image slots. The filter ranks on sign-code
overlap and keeps the closest few.

⚠ **OFF by default.** It ranks using `sign_codes`, which are inherited from text
near a figure rather than read from the figure, so any gain is unproven until
measured over the full 150.

In [ ]:
CFG.figure_relevance_filter = True
_ = ask("What STOP-sign sizes does Table 2B-1 give for a conventional single-lane road?",
        show_scores=True)

## 5. VINE retrieval paths

`retrieve()` is untouched, so the RAG baseline still compares. Two new entry
points sit alongside it:

| method | job | chunks |
|---|---|---|
| `retrieve()` | RAG baseline, unchanged | 6 |
| `retrieve_for_compile()` | find every provision that might hold an obligation | 20 |
| `retrieve_for_obligation()` | everything about one claim, given what is established | 12 |

### 5.1 Per-obligation retrieval

In [ ]:
r = pipeline.retriever
print("new methods:",
      hasattr(r, "retrieve_for_obligation"),
      hasattr(pipeline.store, "fetch_chunks_by_ids"),
      hasattr(pipeline.kg, "chunks_for_section"))

print("4K.04 chunks:", len(pipeline.kg.chunks_for_section("4K.04")))
print("4K.04 cites :", pipeline.kg.sections_cited_by("4K.04"))

res = r.retrieve_for_obligation(
    query="Does this push button installation comply with 4K.04?",
    obligation="the locator tone repeats at 1-second intervals",
    certificates=[{"evidence": [{"type": "section", "id": "4K.04"}]}],
)
print("\nanchors:", res.debug["anchor_sections"], "| anchor chunks:", res.debug["anchor_chunks"])
for c in res.chunks:
    print(f"  {c['section_id']:<9} {c['content_type']:<9} {c.get('text','')[:60]}")

### 5.2 Compile-time retrieval with cross-reference expansion

For each section the top chunks came from, pull in the sections it cites. Only
`cites_section` edges — not `mentions` (2,559) or `depicts` (2,202), which
would flood the results.

Expanded chunks were never ranked, so they get **reserved output slots**
(`expansion_reserved_slots`, default 3). Reserving slots only in the candidate
pool did nothing: the reranker then scored everything together and the expanded
chunks lost again.

In [ ]:
r = pipeline.retriever
q = "minimum sizes for regulatory signs on multi-lane conventional roads"
on  = r.retrieve_for_compile(q)
off = r.retrieve_for_compile(q, expand_cross_references=False)
print("expanded kept:", on.debug["n_expanded_kept"], "of", on.debug["n_chunks"])
print("with    :", sorted({c['section_id'] for c in on.chunks}))
print("without :", sorted({c['section_id'] for c in off.chunks}))
print("gained  :", sorted({c['section_id'] for c in on.chunks} - {c['section_id'] for c in off.chunks}))

### 5.3 Tune the reserved slot count

Every reserved slot is one the search does not get. Pick where the curve flattens.

In [ ]:
r = pipeline.retriever
q = "minimum sizes for regulatory signs on multi-lane conventional roads"
base = {c['section_id'] for c in r.retrieve_for_compile(q, expand_cross_references=False).chunks}
print(f"{'reserved':>9} {'kept':>5} {'secs':>5}  gained / lost")
print(f"{'off':>9} {0:>5} {len(base):>5}")
for n in (2, 3, 5, 8):
    on = r.retrieve_for_compile(q, reserved_slots=n)
    s = {c['section_id'] for c in on.chunks}
    print(f"{n:>9} {on.debug['n_expanded_kept']:>5} {len(s):>5}  +{sorted(s-base)}  -{sorted(base-s)}")

## 6. Table and figure notes

Notes are now chunks of their own, so most of what this section used to probe
by hand is in `chunks.jsonl`. The cells below still run the raw cell extractor,
which is the starting point for turning tables into numbers a calculator can
use — nothing there is decided yet.

The notes themselves carry no printed Standard/Guidance/Option heading, so the
type is inferred from the verb (Section 1C.01: Standard uses only "shall",
Guidance "should", Option "may"). Every such chunk has
`authority_inferred: true` so VINE can tell an inferred type from a printed one.

In [ ]:
import json, collections
rows = [json.loads(l) for l in open(CFG.chunks_jsonl) if l.strip()]
notes = [r for r in rows if r.get("source") in ("table_note", "figure_note")
         and r.get("authority_inferred")]
print(f"notes printed inside crops: {len(notes)}")
print("  from tables :", len({r['parent_id'] for r in notes if r['source']=='table_note'}), "tables")
print("  from figures:", len({r['parent_id'] for r in notes if r['source']=='figure_note'}), "figures")
print("  inferred type:", dict(collections.Counter(r["content_type"] for r in notes)))

print("\nnotes that carry an obligation (shall / should):")
for r in [x for x in notes if x["content_type"] in ("Standard", "Guidance")][:8]:
    print(f"   [{r['content_type']:<8}] {r['text'][:115]}")

The Part 6 typical-application notes are separate: those pages print real Standard/Guidance/Option headings, so nothing is inferred there.

In [ ]:
import json, collections
rows = [json.loads(l) for l in open(CFG.chunks_jsonl) if l.strip()]
ta = [r for r in rows if r["chunk_id"].startswith("MUTCD11e_6P01_TA")]
print(f"Part 6 notes: {len(ta)} items over "
      f"{len({r['parent_id'] for r in ta})} figures (expected 469 over 54)")
print("printed types:", dict(collections.Counter(r["content_type"] for r in ta)))
print("all printed, none inferred:", not any(r["authority_inferred"] for r in ta))
for r in ta[:2] + [x for x in ta if x["chunk_id"].startswith("MUTCD11e_6P01_TA14")][:2]:
    print(f"\n   {r['chunk_id']} [{r['content_type']}]\n   {r['text'][:170]}")

In [ ]:
import json, pickle, subprocess, re, collections
from mrag.table_content import extract_for_crop

# pdftohtml runs for one page, in the shape extract_for_crop expects
def _page_runs(pdf_path, pno):
    xml = subprocess.run(["pdftohtml", "-xml", "-i", "-q", "-stdout",
                          "-f", str(pno), "-l", str(pno), str(pdf_path)],
                         capture_output=True, text=True, errors="ignore").stdout
    fonts = {m.group(1): (float(m.group(2)), re.sub(r"^[A-Z]{6}\+", "", m.group(3)))
             for m in re.finditer(r'<fontspec id="(\d+)" size="([-\d.]+)" family="([^"]*)"', xml)}
    out = []
    for m in re.finditer(r'<text top="(-?\d+)" left="(-?\d+)" width="(-?\d+)" '
                         r'height="(-?\d+)" font="(\d+)">(.*?)</text>', xml, re.S):
        txt = re.sub(r"<[^>]+>", "", m.group(6)).strip()
        if txt:
            size, family = fonts.get(m.group(5), (0.0, "?"))
            out.append((int(m.group(1)), int(m.group(2)), int(m.group(3)),
                        int(m.group(4)), family, size, txt))
    return out

crops = {r["image_path"].split("/")[-1]: r
         for r in (json.loads(l) for l in open(CFG.figures_jsonl) if l.strip())}

for name in ["table_3G-1_p0662.png", "table_2J-2_p0540.png", "table_2C-3_p0193.png"]:
    r = crops.get(name)
    if not r:
        print(f"{name}: not in figures.jsonl"); continue
    d = extract_for_crop(_page_runs(CFG.pdf_path, r["page_pdf"]), r).to_dict()
    print(f"\n{name}: cols={d['n_cols']} rows={len(d['rows'])} notes={len(d['footnotes'])}")
    for row in d["rows"][:2]:
        print("   ", [c["text"] + (f"^{c['footnote_ref']}" if "footnote_ref" in c else "")
                      for c in row][:6])
    for n in d["footnotes"][:4]:
        print(f"   [{n['marker']}] {n['text'][:66]}")

## 7. Run the MUTCD-150 benchmark

In [ ]:
# The runner is a Python API, not a CLI.
import sys
REPO = "/content/Beyond_RAG_repo"
sys.path.insert(0, f"{REPO}/benchmarks/mutcd150/v1")

from mutcd_benchmark_runner import run_benchmark
from mrag.ask import ask
from mrag.config import CFG

BENCH = f"{REPO}/benchmarks/mutcd150/v1/mutcd_benchmark_questions_v1.jsonl"
OUT   = CFG.base_dir / "benchmark_runs"          # Drive/MyDrive/Beyond_RAG/benchmark_runs

paths = run_benchmark(
    CFG=CFG,
    ask_fn=ask,
    questions_path=BENCH,
    output_root=OUT,
    run_id="beyond_rag_002_roundrobin",          # NEW id — 001 used greedy allocation
    models=[{"alias": "fable", "selector": "frontier_claude", "provider": "anthropic"}],
    prompt_style="fewshot",

    # Smoke-test the regression first. Run these three, check TB009 answers
    # instead of abstaining, THEN comment this line out for the full 150.
    #question_ids=["TB008", "TB009", "TB026"],

    resume=True,
)
for k, v in paths.items():
    print(f"{k:<24} {v}")

## 8. Diagnostics

None of these change anything. Run as needed.

### 8.1 What did retrieval return?

In [ ]:
from mrag.retrieval import Retriever

res = pipeline.retriever.retrieve("STOP sign sizes at an all-way stop")

print(f"{len(res.chunks)} chunks\n")
for c in res.chunks:
    print(f"  {c.get('section_id'):<10} {c.get('content_type'):<9} "
          f"p.{c.get('page_printed'):<5} score={c.get('score', 0):.3f}")

print(f"\n{len(res.figures)} figures")
for f in res.figures:
    n = len(f.get("image_paths") or [])
    print(f"  {f.get('figure_id'):<16} sheets={n:<3} source={f.get('source','?')}")

### 8.2 Knowledge graph shape

In [ ]:
import collections
kg = pipeline.kg
print(kg.g.number_of_nodes(), "nodes,", kg.g.number_of_edges(), "edges")
print()
print("node kinds:")
for k, v in collections.Counter(d.get("kind", "?") for _, d in kg.g.nodes(data=True)).most_common():
    print(f"  {k:<14} {v}")
print()
print("edge labels:")
for k, v in collections.Counter(d.get("label", "?") for *_, d in kg.g.edges(data=True)).most_common():
    print(f"  {k:<20} {v}")

### 8.3 Cross-references from a section

In [ ]:
# Sections cross-referenced by a given section (cites_section edges).
target = "2B.04"
refs = set()
for c in (json.loads(l) for l in open(CFG.chunks_jsonl) if l.strip()):
    if c["section_id"] == target:
        refs.update(c.get("section_refs") or [])
print(f"{target} cross-references: {sorted(refs)}")

### 8.4 Is the sparse leg alive?

Expect **0 non-empty** until FlagEmbedding is fixed. BGE-M3 fails to load
(`XLMRobertaModel.__init__() got an unexpected keyword argument 'dtype'` —
FlagEmbedding ≥1.4 against transformers 4.54.1) and `TextEmbedder` falls back
to sentence-transformers, which produces dense vectors only. Retrieval has been
dense-only the whole time, so "hybrid dense + learned sparse" does not yet
describe what runs. This is the next thing to fix.

In [ ]:
import json
s = json.load(open(CFG.cache_dir / "chunks_sparse.json"))
print(len(s), "entries |", sum(1 for x in s if x), "non-empty")

### 8.5 Inspect a table's crops

In [ ]:
from IPython.display import display, Image as IPImage
import json
figs = [json.loads(l) for l in open(CFG.figures_jsonl) if l.strip()]
t = [f for f in figs if f["canonical_id"] == "2B-1" and f["kind"] == "Table"]
t.sort(key=lambda f: f["page_pdf"])
print(f"{len(t)} crops for Table 2B-1\n")
for f in t:
    print(f"pdf p{f['page_pdf']}  printed p{f['page_printed']}  "
          f"sheet={f.get('sheet')}/{f.get('sheet_of')}  {f['image_path'].split('/')[-1]}")
    display(IPImage(filename=f["image_path"], width=560))

### 8.6 Ablations

All six were already run and graded. Re-run only if the pipeline changes underneath them.

In [ ]:
!ls /content/Beyond_RAG_repo/ablations/configs

# NOTE: all six ablations were already run and graded — 900 records, in the
# per-response grading rationales. Published full-credit rates:
#
#   baseline 80.0 | A1 no-router 76.0 | A2 no-VLM-filter 74.7
#   A3 no-graph 76.0 | A4 no-rule-type 74.7 | A5 no-hierarchy 72.7
#   A6 no-reranker 68.7
#
# Only A2, A5 and A6 appear in the submitted paper. A3 (graph) and A4
# (rule type) are the two a reviewer will ask about, and they exist.
#
# Re-run one only if the pipeline changes underneath it:
# !cd /content/Beyond_RAG_repo && python ablations/run_ablation.py \
#     --ablation A3_no_graph --retrieval-only \
#     --output ablations/results/A3_no_graph.json